# Portfolio Optimization - Full Notebook

**Exact Method (MILP) vs Heuristic (Greedy)**  
Aligned comparison with same constraints: K=5 assets, min 5% weight

In [ ]:
import pandas as pd
import pulp
import time
import os
import numpy as np

print("✅ All libraries imported successfully")

## 1. Load Data

In [ ]:
def load_stock_data(folder="./stock_data"):
    all_data = []
    for file in os.listdir(folder):
        if file.endswith("_historical_data.csv"):
            ticker = file.split("_")[0]
            df = pd.read_csv(os.path.join(folder, file), parse_dates=['Date'])
            df.set_index('Date', inplace=True)
            df = df[['Close']].rename(columns={'Close': ticker})
            all_data.append(df)
    
    prices = pd.concat(all_data, axis=1).dropna()
    return prices

# ====================== LOAD DATA ======================
prices = load_stock_data()
returns = prices.pct_change().dropna()

print(f"✅ Data loaded successfully!")
print(f"   → {prices.shape[0]} trading days")
print(f"   → {prices.shape[1]} stocks: {list(prices.columns)}")

## 2. Aligned MILP (Exact Method)

In [ ]:
def run_aligned_milp(returns, K=5, min_weight=0.05, max_weight=0.30):
    start_time = time.time()
    tickers = returns.columns.tolist()
    mean_returns = returns.mean()

    model = pulp.LpProblem("Aligned_MILP_Portfolio", pulp.LpMaximize)

    x = pulp.LpVariable.dicts("weight", tickers, 0, 1)
    y = pulp.LpVariable.dicts("select", tickers, cat='Binary')

    # Objective: Maximize expected return
    model += pulp.lpSum(mean_returns[i] * x[i] for i in tickers)

    # Constraints
    model += pulp.lpSum(x[i] for i in tickers) == 1, "Budget"
    model += pulp.lpSum(y[i] for i in tickers) == K, "Cardinality"

    for i in tickers:
        model += x[i] <= y[i], f"link_{i}"
        model += x[i] >= min_weight * y[i], f"min_{i}"
        model += x[i] <= max_weight, f"max_{i}"

    status = model.solve(pulp.PULP_CBC_CMD(msg=False, timeLimit=60))
    runtime = time.time() - start_time

    weights_dict = {i: round(x[i].value(), 6) for i in tickers if x[i].value() > 0.001}
    selected = [i for i in tickers if y[i].value() > 0.5]

    port_return = sum(mean_returns[i] * x[i].value() for i in tickers)
    weights_series = pd.Series({i: x[i].value() for i in tickers})
    port_mad = (returns @ weights_series).abs().mean()

    return {
        'method': 'MILP',
        'selected': selected,
        'weights': weights_dict,
        'expected_return': round(port_return, 6),
        'mad_risk': round(port_mad, 6),
        'runtime': round(runtime, 4),
        'status': pulp.LpStatus[status]
    }


milp_result = run_aligned_milp(returns, K=5, min_weight=0.05, max_weight=0.30)

print("\n" + "="*70)
print("               MILP RESULTS (K=5)")
print("="*70)
print("Selected:", milp_result['selected'])
print("\nWeights:")
for t, w in milp_result['weights'].items():
    print(f"  {t:6} : {w:.4f} ({w*100:5.1f}%)")
print(f"\nExpected Return : {milp_result['expected_return']:.6f}")
print(f"MAD Risk        : {milp_result['mad_risk']:.6f}")
print(f"Runtime         : {milp_result['runtime']} sec")

## 3. Aligned Greedy Heuristic

In [ ]:
def run_greedy(returns, K=5, min_weight=0.05):
    start_time = time.time()
    
    mean_returns = returns.mean()
    std_returns = returns.std()
    scores = mean_returns / std_returns
    
    # Select top K stocks
    selected = scores.nlargest(K).index.tolist()
    
    # Minimum weight first
    weights = pd.Series(min_weight, index=selected)
    remaining = 1.0 - (K * min_weight)
    
    # Distribute remaining weight proportionally
    extra = (scores[selected] / scores[selected].sum()) * remaining
    weights += extra
    weights = weights / weights.sum()
    
    runtime = time.time() - start_time
    
    port_return = (weights * mean_returns[selected]).sum()
    port_mad = (returns[selected] @ weights).abs().mean()
    
    return {
        'method': 'Greedy',
        'selected': selected,
        'weights': {t: round(float(w), 6) for t, w in weights.items()},
        'expected_return': round(port_return, 6),
        'mad_risk': round(port_mad, 6),
        'runtime': round(runtime, 4)
    }


greedy_result = run_greedy(returns, K=5, min_weight=0.05)

print("\n" + "="*70)
print("             GREEDY RESULTS (K=5)")
print("="*70)
print("Selected:", greedy_result['selected'])
print("\nWeights:")
for t, w in greedy_result['weights'].items():
    print(f"  {t:6} : {w:.4f} ({w*100:5.1f}%)")
print(f"\nExpected Return : {greedy_result['expected_return']:.6f}")
print(f"MAD Risk        : {greedy_result['mad_risk']:.6f}")
print(f"Runtime         : {greedy_result['runtime']} sec")

## 4. Comparison Table

In [ ]:
comparison = pd.DataFrame({
    'Metric': ['Expected Daily Return', 'MAD Risk', 'Runtime (seconds)', 'Number of Assets'],
    'MILP (Exact)': [
        milp_result['expected_return'],
        milp_result['mad_risk'],
        milp_result['runtime'],
        len(milp_result['selected'])
    ],
    'Greedy (Heuristic)': [
        greedy_result['expected_return'],
        greedy_result['mad_risk'],
        greedy_result['runtime'],
        len(greedy_result['selected'])
    ]
})

print("\n" + "="*80)
print("               FINAL COMPARISON: MILP vs GREEDY")
print("="*80)
print(comparison.round(6).to_string(index=False))

print("\nSelected Assets:")
print(f"MILP   → {milp_result['selected']}")
print(f"Greedy → {greedy_result['selected']}")